# Event selection — slices per stage

Reads **chunk pickles** from `event_selection_chunk.py` (map phase), sums per-sample accumulators, and prints how many reconstructed **slices** survive each pipeline stage.

**What is stored in the pickles**

- `bar[stage_key]["topology"]` — per-stage sum of `pot_weight` (≈ slice count when weights are 1 per event). Filled for MC, intime, offbeam, dirt, and data at stages with `save_for_breakdown=True`.
- `eff` (MC only) — efficiency histograms; `n_at_stage_int` matches the same total for MC when topology categories partition all events.

Counts here are **before** aggregate POT scaling (`apply_global_exposure_scales`). Re-run the chunked map if you changed breakdown stages (e.g. `allreco` / `is_clear_cosmic` / data).

**Setup:** set `CHUNKS_DIR` to your `.../chunks` directory (same as `run_event_selection_chunked.sh`).

In [ ]:
from __future__ import annotations

import glob
import os
import sys
from pathlib import Path

import pandas as pd

_cwd = Path.cwd().resolve()
CAFPYANA_ROOT = None
for _p in [_cwd, *_cwd.parents]:
    if (_p / "pyanalib").is_dir() and (_p / "analysis_village").is_dir():
        CAFPYANA_ROOT = _p
        break
if CAFPYANA_ROOT is None:
    raise RuntimeError("Could not find cafpyana repo root (need pyanalib/ + analysis_village/).")
sys.path.insert(0, str(CAFPYANA_ROOT))

from analysis_village.numucc_1p0pi.dataset_locations import default_event_selection_work_root
from analysis_village.numucc_1p0pi.selection_framework import aggregate_chunk_files

SAMPLES = ("mc", "data", "intime", "offbeam", "dirt")

# --- edit: directory with mc__*.pkl, data__*.pkl, ... ---
CHUNKS_DIR = Path(os.environ.get("CHUNKS_DIR", default_event_selection_work_root() / "chunks"))
CHUNKS_DIR = CHUNKS_DIR.expanduser().resolve()
print("CHUNKS_DIR =", CHUNKS_DIR)
if not CHUNKS_DIR.is_dir():
    raise FileNotFoundError(f"CHUNKS_DIR does not exist: {CHUNKS_DIR}")

In [ ]:
def chunk_files_for_sample(chunks_dir: Path, sample: str) -> list[str]:
    pattern = str(chunks_dir / f"{sample}__*.pkl")
    paths = sorted(glob.glob(pattern))
    if not paths:
        print(f"WARNING: no pickles for sample={sample!r} under {pattern}")
    return paths


def load_aggregated_per_sample(chunks_dir: Path) -> dict[str, dict]:
    """Sum chunk pickles per sample (intrinsic weights, no global POT scale)."""
    out = {}
    for sample in SAMPLES:
        files = chunk_files_for_sample(chunks_dir, sample)
        if not files:
            continue
        out[sample] = aggregate_chunk_files(files)
        print(f"{sample}: {len(files)} chunk pickle(s)")
    return out


def stage_counts_table(per_sample: dict[str, dict], breakdown_type: str = "topology") -> pd.DataFrame:
    """Rows = stages, columns = sample; values = POT-weighted slice totals."""
    if not per_sample:
        raise ValueError("no samples loaded")

    stage_keys = per_sample[next(iter(per_sample))]["stage_keys"]
    stage_labels = dict(
        zip(
            per_sample[next(iter(per_sample))]["stage_keys"],
            per_sample[next(iter(per_sample))]["stage_labels"],
        )
    )

    rows = []
    for sk in stage_keys:
        row = {"stage_key": sk, "stage_label": stage_labels.get(sk, sk)}
        for sample in SAMPLES:
            agg = per_sample.get(sample)
            if agg is None or sk not in agg.get("bar", {}):
                row[sample] = float("nan")
                continue
            by_bt = agg["bar"][sk]
            if breakdown_type not in by_bt:
                row[sample] = float("nan")
                continue
            row[sample] = by_bt[breakdown_type].total_count(sample)
        rows.append(row)

    df = pd.DataFrame(rows).set_index(["stage_key", "stage_label"])
    return df


per_sample = load_aggregated_per_sample(CHUNKS_DIR)

In [ ]:
counts = stage_counts_table(per_sample)
counts_int = counts.round(0).astype("Int64")

print("Selected slices per stage (sum of pot_weight over chunks; intrinsic, unscaled)\n")
try:
    display(counts_int)
except NameError:
    print(counts_int.to_string())

summary = counts_int.reset_index().set_index("stage_label")
summary.index.name = "stage"
print("\nSame table (stage label index):\n")
try:
    display(summary)
except NameError:
    print(summary.to_string())

In [ ]:
# Optional: cross-check MC bar totals vs efficiency accumulator (should agree)
mc = per_sample.get("mc")
if mc and mc.get("eff"):
    first_stage = next(iter(mc["eff"]))
    var0 = next(iter(mc["eff"][first_stage]))
    eff_rows = []
    for sk in mc["stage_keys"]:
        bar_n = float("nan")
        if sk in mc.get("bar", {}) and "topology" in mc["bar"][sk]:
            bar_n = mc["bar"][sk]["topology"].total_count("mc")
        if sk not in mc["eff"] or var0 not in mc["eff"][sk]:
            continue
        eff_rows.append(
            {
                "stage_key": sk,
                "bar_topology": bar_n,
                "eff_n_at_stage_int": mc["eff"][sk][var0].n_at_stage_int,
            }
        )
    if eff_rows:
        chk = pd.DataFrame(eff_rows).set_index("stage_key")
        print("MC: bar vs eff total slice weight\n")
        display(chk)